# FoodVision on the 101 Types of Foods

Each Types of Food contains 1000 Pictures in total 101000 Pictures.
Taken from Pytorch Food101 Dataset 

We are making FoodVision model from pretrained Effcientnet_b4 with all base layer frozen

It taken upto 10-12 GB of Internet data and 20-30 GB of free space

I am using 
Programming Language: Python
Framework: Pytorch 
RAM: 24GB
Processer: Intel i5 H 12Gen H Grade
GPU: Nvidia RTX 2050 Mobile

It consits of total Trainable parameters is 181,093.
Accuracy: 67-68% on 5 Epochs



Total params: 17,729,709
Trainable params: 181,093
Non-trainable params: 17,548,616
Total mult-adds (Units.GIGABYTES): 24.04

Input size (MB): 9.63
Forward/backward pass size (MB): 4359.81
Params size (MB): 70.92
Estimated Total Size (MB): 4440.36

Let's Start Code

Now, install all dependences for our model and import all library

In [87]:
try:
    import torch
    import torchvision
    import torch.nn as nn
except:
    print("Library Not Found\nDownloading...")
    !python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
    import torch
    import torchvision
    import torch.nn as nn
try:
    import matplotlib.pyplot as plt
    from tqdm.auto import tqdm
    from typing import Dict, List, Tuple
    from IPython.display import clear_output
    from timeit import default_timer as timer
    import os
    import random
    import gradio as gr
    from torchinfo import summary
except:
    print("Library Not Found\nDownloading...")
    !python -m pip install matplotlib tqdm typing IPython torchinfo gradio
    import matplotlib.pyplot as plt
    from tqdm.auto import tqdm
    from typing import Dict, List, Tuple
    from IPython.display import clear_output
    from timeit import default_timer as timer
    import os
    import random
    import gradio as gr
    from torchinfo import summary
try:
    !cloudflared --version
except:
    print("Cloudflared Not Found\nDownloading...")
    !winget install --id Cloudflare.cloudflared

cloudflared version 2026.8.3 (built 2026-08-31T02:48 UTC)


Check this our torch is install in correct device. Otherwise, It shows device == cpu then check for correct installation process of the torch from the any Ai

In [88]:
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("GPU not found. Please check your CUDA installation.")
else:
    print("GPU found. Using CUDA for computations.")

GPU found. Using CUDA for computations.


#### Now, We installing our pretrained model efficientNet_b4 for training this dataset

Add any compitable pretrained model from torchvision pretrained model and freeze the base layers of the model 

In [89]:
def create_model(model:torchvision.models=torchvision.models.efficientnet_b4,weights:torchvision.models=torchvision.models.EfficientNet_B4_Weights.DEFAULT,num_classes:int=101,
                 seed:int=42):
    """Creates an feature extractor model and transforms.
    
        Args:
            model (torchvision.models, optional): model architecture. Defaults to torchvision.models.efficientnet_b4.
            weights (torchvision.models, optional): pretrained weights for the model. Defaults to torchvision.models
            num_classes (int, optional): number of classes in the classifier head. 
                Defaults to 101.
            seed (int, optional): random seed value. Defaults to 42.
    
        Returns:
            model (torch.nn.Module): feature extractor model. 
            transforms (torchvision.transforms): image transforms.
        """
    model = model(weights=weights).to(device) # 1. Load pretrained model with weights
    transform = weights.transforms() #2. Load transforms for the model
        # 3. Freeze all layers in base model
    for param in model.parameters():
        param.requires_grad = False
    
        # 4. Change classifier head with random seed for reproducibility
    torch.manual_seed(seed)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1792, out_features=num_classes),
        )
        
    return model, transform

In [90]:
model,transform = create_model(model=torchvision.models.efficientnet_b4,weights=torchvision.models.EfficientNet_B4_Weights.DEFAULT,num_classes=101,seed=42)
model.classifier

Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1792, out_features=101, bias=True)
)

Now, Downloading the Dataset from torchvision Dataset

In [91]:
from torchvision import datasets
train_data = datasets.Food101(
    root="data",
    split="train",
    transform=transform,
    download=True
) # Training data

test_data = datasets.Food101(
    root="data",
    split="test",
    transform=transform,
    download=True
) # Testing data
from pathlib import Path
image_path = Path("data/food-101/images") # Setting the path to the images folder

In [92]:
def create_dataloaders(train_data,
                       test_data,
                       tranasforms:torchvision.transforms.Compose,
                       batch_size:int=32,
                       num_workers:int=8):
    """Creates dataloaders for training and testing datasets.
    
        Args:
            train_directory (str): path to the training dataset.
            test_directory (str): path to the testing dataset.
            transforms (torchvision.transforms.Compose): image transforms.
            batch_size (int, optional): number of samples per batch. Defaults to 32.
            num_workers (int, optional): number of subprocesses to use for data loading. Defaults to 8.
        Returns:
        It will give tuple of train_dataloader, test_dataloader and class_names.
            train_dataloader (torch.utils.data.DataLoader): dataloader for training dataset.
            test_dataloader (torch.utils.data.DataLoader): dataloader for testing dataset.
            class_names (list): list of class names in the dataset.
            Example:
                train_dataloader, test_dataloader, class_names = create_dataloaders(train_directory="data/food-101/images/train",
                                                                                     test_directory="data/food-101/images/test",
                                                                                     transforms=transform,
                                                                                     batch_size=32,
                                                                                     num_workers=4)
        """
    train_dataloader = torch.utils.data.DataLoader(
        train_data,
        shuffle=True,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True
    ) # Training dataloader
    test_dataloader = torch.utils.data.DataLoader(
        test_data,
        shuffle=False,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True
    ) # Testing dataloader
    return train_dataloader, test_dataloader, train_data.classes


Create Dataloader for Testing and Training

In [93]:
train_dataloader, test_dataloader, class_names = create_dataloaders(
    train_data=train_data,
    test_data=test_data,
    tranasforms=transform,
    batch_size=32,
    num_workers=4
)

In [94]:
summary(model, input_size=(16, 3, 224, 224), device=device)

Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [16, 101]                 --
├─Sequential: 1-1                                       [16, 1792, 7, 7]          --
│    └─Conv2dNormActivation: 2-1                        [16, 48, 112, 112]        --
│    │    └─Conv2d: 3-1                                 [16, 48, 112, 112]        (1,296)
│    │    └─BatchNorm2d: 3-2                            [16, 48, 112, 112]        (96)
│    │    └─SiLU: 3-3                                   [16, 48, 112, 112]        --
│    └─Sequential: 2-2                                  [16, 24, 112, 112]        --
│    │    └─MBConv: 3-4                                 [16, 24, 112, 112]        (2,940)
│    │    └─MBConv: 3-5                                 [16, 24, 112, 112]        (1,206)
│    └─Sequential: 2-3                                  [16, 32, 56, 56]          --
│    │    └─MBConv: 3-6                    

Now, Create loss_fn, optimizer

In [95]:
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1) # Loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.001,weight_decay=0.0001) # Optimizer

Let's Start the model train

In [96]:
def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epochs: int = 5,
    device: torch.device = torch.device("cuda"),
    model_name: str = "food101_model.pth",
    model_path: Path = Path("models/"),
    resume: bool = False
):

    """
    Trains a PyTorch model on a given dataset.
    Args:
        model (torch.nn.Module): The PyTorch model to train.
        train_dataloader (torch.utils.data.DataLoader): DataLoader for the training dataset.
        test_dataloader (torch.utils.data.DataLoader): DataLoader for the testing dataset.
        loss_fn (torch.nn.Module): Loss function to use for training.
        optimizer (torch.optim.Optimizer): Optimizer to use for training.
        epochs (int, optional): Number of epochs to train for. Defaults to 5.
        device (torch.device, optional): Device to train on. Defaults to torch.device("cuda").
        model_name (str, optional): Name of the model file to save. Defaults to "food101_model.pth".
        model_path (Path, optional): Path to save the model. Defaults to Path("models/").
        resume (bool, optional): Whether to resume training from a checkpoint. Defaults to False.
    Returns:
        results (dict): A dictionary containing training and testing loss and accuracy for each epoch.
    """
    
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    model.to(device)

    # ==========================================
    # LOAD CHECKPOINT IF RESUMING
    # ==========================================

    start_epoch = 0

    if resume:

        checkpoint = torch.load(
            model_path / model_name,
            map_location=device
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"]

        print(f"Resuming training from epoch {start_epoch + 1}")


    # ==========================================
    # TRAINING
    # ==========================================

    for epoch in range(start_epoch, epochs):

        # ======================================
        # TRAIN
        # ======================================

        model.train()

        train_loss, train_acc = 0, 0
        total_images = 0

        train_bar = tqdm(
            train_dataloader,
            desc=f"Epoch {epoch + 1}/{epochs} [Train]",
            leave=True
        )

        for X, y in train_bar:

            X, y = X.to(device), y.to(device)

            # Forward
            y_pred = model(X)

            # Loss
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Accuracy
            y_pred_class = y_pred.argmax(dim=1)

            train_acc += (
                (y_pred_class == y).sum().item()
                / len(y_pred)
            )

            # Image counter
            total_images += len(X)

            train_bar.set_postfix(
                images=f"{total_images}/{len(train_dataloader.dataset)}"
            )


        # Average training metrics
        train_loss /= len(train_dataloader)
        train_acc /= len(train_dataloader)


        # ======================================
        # TEST
        # ======================================

        model.eval()

        test_loss, test_acc = 0, 0
        total_images = 0

        test_bar = tqdm(
            test_dataloader,
            desc=f"Epoch {epoch + 1}/{epochs} [Test]",
            leave=True
        )

        with torch.inference_mode():

            for X, y in test_bar:

                X, y = X.to(device), y.to(device)

                # Forward
                test_pred_logits = model(X)

                # Loss
                loss = loss_fn(
                    test_pred_logits,
                    y
                )

                test_loss += loss.item()

                # Accuracy
                test_pred_labels = test_pred_logits.argmax(
                    dim=1
                )

                test_acc += (
                    (test_pred_labels == y).sum().item()
                    / len(test_pred_labels)
                )

                # Image counter
                total_images += len(X)

                test_bar.set_postfix(
                    images=f"{total_images}/{len(test_dataloader.dataset)}"
                )


        # Average testing metrics
        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)


        # ======================================
        # EPOCH RESULT
        # ======================================

        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )


        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)


        # ======================================
        # SAVE CHECKPOINT
        # ======================================
        Path(model_path).mkdir(parents=True, exist_ok=True)
        torch.save({

            "epoch": epoch + 1,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "train_loss":
                train_loss,

            "train_acc":
                train_acc,

            "test_loss":
                test_loss,

            "test_acc":
                test_acc,

        }, model_path / model_name)


        print(
            f"Model saved: Epoch {epoch + 1}"
        )


    return results

Ploting loss curves from results Dict values

In [97]:
def plot_loss_curves(results):
    """Plots training curves of a results dictionary.

    Args:
        results (dict): dictionary containing list of values, e.g.
            {"train_loss": [...],
             "train_acc": [...],
             "test_loss": [...],
             "test_acc": [...]}
    """
    loss = results["train_loss"]
    test_loss = results["test_loss"]

    accuracy = results["train_acc"]
    test_accuracy = results["test_acc"]

    epochs = range(len(results["train_loss"]))

    plt.figure(figsize=(15, 7))

    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label="train_loss")
    plt.plot(epochs, test_loss, label="test_loss")
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()

    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label="train_accuracy")
    plt.plot(epochs, test_accuracy, label="test_accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()


Here, We train or retrain our model if required

Make Sure if You already trained your model so skip this Cell from run, I required only 5 Epoch for best Results

In [ ]:
model_results = train(model=model,
                      train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        loss_fn=loss_fn,
                        optimizer=optimizer,
                        epochs=5,
                        device=device,
                        resume=False,
                        model_name="food101_model.pth",
                        model_path=Path("models/")
)
plot_loss_curves(model_results)

Create Predicting Function for predit our custom data from real world data

In [ ]:
def predict(img) -> Tuple[Dict, float]:
    """Transforms and performs a prediction on img and returns prediction and time taken.
    """
    # Start the timer
    start_time = timer()
    checkpoint = torch.load("models/food101_model.pth", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    # Transform the target image and add a batch dimension
    img = transform(img).unsqueeze(0)
    img = img.to(device)
    # Put model into evaluation mode and turn on inference mode
    model.eval()
    with torch.inference_mode():
        # Pass the transformed image through the model and turn the prediction logits into prediction probabilities
        pred_probs = torch.softmax(model(img), dim=1)
    
    # Create a prediction label and prediction probability dictionary for each prediction class (this is the required format for Gradio's output parameter)
    pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}
    
    # Calculate the prediction time
    pred_time = round(timer() - start_time, 5)
    
    # Return the prediction dictionary and prediction time 
    return pred_labels_and_probs, pred_time

Create Example List for Trial

In [ ]:
data_path = Path("data/food-101/images")
def create_example(data_path: Path=Path("data/food-101/images"), num_examples: int = 5) -> Tuple[List[Path], List[str]]:
    """Creates a list of example images and their corresponding labels from the dataset.

    Args:
        data_path (Path): Path to the dataset directory.
        num_examples (int, optional): Number of examples to create. Defaults to 5.

    Returns:
        Tuple[List[Path], List[str]]: A tuple containing a list of example image paths and a list of corresponding labels.
    """
    if num_examples > len(list(data_path.iterdir())):
        raise ValueError(f"num_examples ({num_examples}) is greater than the number of food classes in the dataset ({len(list(data_path.iterdir()))}). Please choose a smaller value for num_examples.")
    example_images = []
    example_labels = []
    
    # Get a list of all food class directories
    food_classes = [folder for folder in data_path.iterdir() if folder.is_dir()]
    
    # Randomly sample food classes
    sampled_classes = random.sample(food_classes, k=num_examples)
    
    for food_class in sampled_classes:
        # Get all images in the food class directory
        images = list(food_class.glob("*.jpg"))
        
        # Randomly select one image from the class
        random_image = random.choice(images)
        
        # Append the image path and label to the lists
        example_images.append(random_image)
        example_labels.append(food_class.name)
    
    return example_images, example_labels

In [ ]:
example_images, example_labels = create_example(data_path=data_path, num_examples=5)

Create a Hosting Interface doing the prediction on our model

In [ ]:
def host_Interface(title:str="FoodVision",description:str="An EfficientNetB4 feature extractor computer vision model to classify images of food as pizza, steak or sushi.",article:str="Created at Food101_effnetb4.",
                   server_port:int=1000):
    """Hosts the Gradio interface on a public URL using Cloudflare Tunnel.

    Args:
        title (str): Title of the Gradio interface.
        description (str): Description of the Gradio interface.
        article (str): Article text for the Gradio interface.
    """
    demo = gr.Interface(fn=predict, # mapping function from input to output
                        inputs=gr.Image(type="pil"), # what are the inputs?
                        examples=example_images, # example images to show in the demo
                        example_labels=example_labels, # example labels for the example images
                        outputs=[gr.Label(num_top_classes=3, label="Predictions"), # what are the outputs?
                                 gr.Number(label="Prediction time (s)")], # our fn has two outputs, therefore we have two outputs
                        title=title,
                        description=description,
                        article=article)
    server_port = random.randint(1000, 9999)  # Generate a random port number between 1000 and 9999
    demo.launch(server_name="127.0.0.1",server_port=server_port) # generate a publically shareable URL?
    print(f"Gradio interface is running on http://localhost:{server_port}.")
    print(f"To create a public URL using Cloudflare Tunnel, run the following command in your terminal:\n\ncloudflared tunnel --url http://localhost:{server_port}")

Use The Funtion to Create Interface and You Can Share on other to use your model. Note: ClouldFlare termination cause you website not reachable. So, Keep run you system to keep alive you website

In [ ]:
host_Interface(title="FoodVision",description="An EfficientNetB4 feature extractor computer vision model to classify images of food as pizza, steak or sushi.",article="Created at Food101_effnetb4.",
               )

* Running on local URL:  http://127.0.0.1:7407
* To create a public link, set `share=True` in `launch()`.


Gradio interface is running on http://localhost:7407.
To create a public URL using Cloudflare Tunnel, run the following command in your terminal:

cloudflared tunnel --url http://localhost:7407


<img src="https://raw.githubusercontent.com/subham107-py/Foodvision_101/main/Sample_Images/cloudflared_host.png" alt="How to Host clouldflare website" width="1600" height="700">

Copy the above code for Hosting the website

#### This is the preview of the website

<img src="https://raw.githubusercontent.com/subham107-py/Foodvision_101/main/Sample_Images/localhost_gradio_preview.png" alt="Website Preview" width="1600" height="700">

# Create Your Model

You Can make your Model from this easily

Make Sure Upper all cell should run Previously 

In [ ]:
# Use only Pytorch Pretrained Weights for the model and transforms
model, transform = create_model()  # Create the model and transforms
train_data = datasets.Food101(
    root="data",
    split="train",
    transform=transform,
    download=True
) # Training data

test_data = datasets.Food101(
    root="data",
    split="test",
    transform=transform,
    download=True
) # Testing data
image_path = Path("data/food-101/images") # Image path to the images folder
train_dataloader, test_dataloader, class_names = create_dataloaders() # Create the dataloaders

In [ ]:
model_results = train() # Train the model and get the results
plot_loss_curves(model_results) # Plot the loss curves for the model results

In [ ]:
example_images, example_labels = create_example() # Create example images and labels for the Gradio interface
host_Interface() # Host the Gradio interface with the example images and labels